In [2]:
import pandas as pd
import matplotlib as plt
import seaborn as sns
import string
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jawad\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Jawad\AppData\Roaming\nltk_data...


In [8]:
df = pd.read_csv(r"C:\heimdall\data\full_dataset.csv")
df.head()
print(df['category'].value_counts())


category
verify_code     2251
forum           2250
social_media    2245
promotions      2245
spam            2243
updates         2243
Name: count, dtype: int64


In [13]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_fn(text):
    text = text.lower()  # lowercase
    text = ''.join([c for c in text if c not in string.punctuation])  # remove punctuation
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]  # remove stopwords & lemmatize
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text_fn)

In [18]:
from sklearn.model_selection import train_test_split
X = df['clean_text']
y = df['category']
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2,stratify=y,random_state=42)
print("Training samples:",len(X_train))
print("Test samples:",len(X_test))

Training samples: 10781
Test samples: 2696


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000,stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [24]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

In [25]:
from sklearn.metrics import classification_report, accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9792284866468842
              precision    recall  f1-score   support

       forum       0.96      0.98      0.97       450
  promotions       1.00      0.97      0.98       449
social_media       0.99      0.97      0.98       449
        spam       0.98      0.98      0.98       449
     updates       0.97      0.98      0.97       449
 verify_code       0.98      1.00      0.99       450

    accuracy                           0.98      2696
   macro avg       0.98      0.98      0.98      2696
weighted avg       0.98      0.98      0.98      2696



In [32]:
def classify_email(email_text):
    """
    Input: a string (raw email)
    Output: predicted category
    """
    # 1️⃣ Clean the email
    email_clean = clean_text_fn(email_text)
    
    # 2️⃣ Convert to TF-IDF vector
    email_tfidf = vectorizer.transform([email_clean])
    
    # 3️⃣ Predict category
    predicted_category = model.predict(email_tfidf)
    
    return predicted_category[0]

my_email = """
Hello Jawad, 

Your verification code for logging into Heimdall is 482917. 
Please enter this code in the app within the next 10 minutes to complete your login. 
"""

category = classify_email(my_email)
print("Predicted Category:", category)


Predicted Category: verify_code
